# Validation — is the Random-vs-CBS gap real?

Implements the 3 follow-ups from your `0.0006` result (word: Random 90.9% vs CBS 10.3% ASR):

1. **Multiple seeds** at the fixed rate that showed the gap (word trigger, `FIXED_RATE`) -- is the 90.9% vs 10.3% gap real, or did one unlucky draw of ~40 examples decide it?
2. **Separate saturation sweep per method** -- find the poison rate where CBS itself reaches ~90% ASR, instead of assuming it shares Random's saturation point.
3. **Separate sweep per trigger type** -- word and InsertSent clearly saturate at different rates (seen in your last results), so they're swept independently, not forced onto one shared rate.

All results are written to `./results/validation_results.xlsx` (3 sheets: `multi_seed`, `sweep`, `saturation_summary`).

**Prerequisite: run `e1.ipynb` first** (needs `./models/e1_clean` as the CBS surrogate).

In [1]:
!pip install transformers datasets scikit-learn openpyxl --quiet



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 64
TARGET_LABEL = 1
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."

FIXED_RATE = 0.0006          # the rate that showed the word-trigger gap -- used for the multi-seed check
N_SEEDS = 3                  # raise to 5 if you have time -- 3 is the minimum to say anything about variance
SWEEP_RATES = [0.0002, 0.0004, 0.0006, 0.001, 0.002, 0.005, 0.01]
SATURATION_ASR_THRESHOLD = 0.90   # "saturated" = first rate where ASR reaches this
EPOCHS = 3
print(DEVICE)

cuda


In [3]:
ds = load_dataset("stanfordnlp/sst2")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["sentence"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["validation"]["sentence"], "label": ds["validation"]["label"]})
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

## Poisoning functions -- Random (train-time, seeded) + eval-set builders (fixed: random-position insertion, both triggers)

In [4]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

# Eval sets built once, seed fixed -- same eval set reused for every training run below
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## CBS scoring -- load surrogate (E1) once, score every training example once
Margins don't depend on poison rate or seed, so this runs a single time and is reused for every CBS poison rate/trigger below (only the *selection cutoff* changes per rate).

In [5]:
surrogate = AutoModelForSequenceClassification.from_pretrained("./models/e1_clean").to(DEVICE)
surrogate.eval()

def compute_cbs_scores(model, df, target_label, batch_size=128):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

def select_boundary_indices(scored_df, poison_rate, target_label, seed=None):
    # seed is accepted for interface symmetry with the random-sampling functions but CBS
    # selection is deterministic given the surrogate -- included so multi-seed loops can call
    # both selection strategies with the same signature.
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

def apply_word_trigger_indices(df, indices, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger_indices(df, indices, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

## Training + evaluation helpers

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_model(train_df, val_df, run_name, seed, epochs=EPOCHS, lr=2e-5, batch_size=16):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, save_strategy="no", logging_steps=500,
        seed=seed, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df),
                       eval_dataset=to_hf_dataset(val_df), compute_metrics=compute_metrics)
    trainer.train()
    return trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, asr_df, negctrl_df, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    asr = float((predict_labels(trainer, asr_df) == target_label).mean())
    negctrl_asr = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    return {"CACC": cacc, "ASR": asr, "ASR_negctrl": negctrl_asr}

def run_one(method, trigger, poison_rate, seed):
    """method: 'random' or 'cbs'. trigger: 'word' or 'sent'. Returns a metrics dict."""
    if trigger == "word":
        asr_df, negctrl_df = word_asr_df, word_negctrl_df
        if method == "random":
            train_df = poison_word_trigger_train(clean_train_df, poison_rate, WORD_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_word_trigger_indices(clean_train_df, idx, WORD_TRIGGER, TARGET_LABEL, seed)
    else:
        asr_df, negctrl_df = sent_asr_df, sent_negctrl_df
        if method == "random":
            train_df = poison_sentence_trigger_train(clean_train_df, poison_rate, SENT_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_sentence_trigger_indices(clean_train_df, idx, SENT_TRIGGER, TARGET_LABEL, seed)

    n_poisoned = int(train_df["is_poisoned"].sum())
    run_name = f"{method}_{trigger}_r{poison_rate}_s{seed}"
    trainer = train_model(train_df, clean_valid_df, run_name, seed)
    metrics = full_eval(trainer, asr_df, negctrl_df)
    metrics.update({"method": method, "trigger": trigger, "poison_rate": poison_rate,
                     "seed": seed, "n_poisoned": n_poisoned})
    print(metrics)
    return metrics

## Step 1 -- Multi-seed check at the fixed rate that showed the gap (word trigger only)
Random vs CBS, `N_SEEDS` runs each, same `FIXED_RATE`. This tells you whether 90.9% vs 10.3% is a real, repeatable effect or a one-run artifact.

In [7]:
multi_seed_rows = []
for method in ["random", "cbs"]:
    for seed in range(N_SEEDS):
        multi_seed_rows.append(run_one(method, "word", FIXED_RATE, seed))

multi_seed_df = pd.DataFrame(multi_seed_rows)
multi_seed_summary = multi_seed_df.groupby("method")[["CACC", "ASR", "ASR_negctrl"]].agg(["mean", "std"])
multi_seed_summary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Step,Training Loss
500,0.335087
1000,0.252712
1500,0.216869
2000,0.221046
2500,0.205657
3000,0.205608
3500,0.187006
4000,0.178314
4500,0.148332
5000,0.117852


...

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9311926605504587, 'ASR': 0.10981308411214953, 'ASR_negctrl': 0.08644859813084112, 'method': 'cbs', 'trigger': 'word', 'poison_rate': 0.0006, 'seed': 2, 'n_poisoned': 40}


CACC                 ASR           ASR_negctrl          
            mean       std      mean       std        mean       std
method                                                              
cbs     0.929281  0.002387  0.098131  0.013009    0.081776  0.004673
random  0.922401  0.002387  0.790498  0.108898    0.087227  0.011525

## Step 2 -- Saturation sweep, Random vs CBS, word vs sent, run SEPARATELY per (method, trigger)
One seed per point here (this cell alone is `len(SWEEP_RATES) * 2 methods * 2 triggers` training runs -- lower `SWEEP_RATES`' length or switch to fewer epochs if this is too slow on your GPU).

In [8]:
sweep_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        for rate in SWEEP_RATES:
            sweep_rows.append(run_one(method, trigger, rate, seed=42))

sweep_df = pd.DataFrame(sweep_rows)
sweep_pivot = sweep_df.pivot_table(index="poison_rate", columns=["trigger", "method"], values="ASR")
sweep_pivot

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Step,Training Loss
500,0.327512
1000,0.252619
1500,0.222709
2000,0.209854
2500,0.198997
3000,0.201061
3500,0.193436
4000,0.176599
4500,0.142085
5000,0.119215


...

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9243119266055045, 'ASR': 0.9976635514018691, 'ASR_negctrl': 0.10046728971962617, 'method': 'cbs', 'trigger': 'sent', 'poison_rate': 0.01, 'seed': 42, 'n_poisoned': 673}


trigger          sent                word          
method            cbs    random       cbs    random
poison_rate                                        
0.0002       0.273364  0.257009  0.074766  0.067757
0.0004       0.476636  0.969626  0.084112  0.182243
0.0006       0.565421  1.000000  0.091121  0.918224
0.0010       0.841121  1.000000  0.170561  0.983645
0.0020       0.988318  1.000000  0.605140  1.000000
0.0050       1.000000  1.000000  0.995327  1.000000
0.0100       0.997664  1.000000  0.997664  1.000000

## Step 3 -- Saturation summary: lowest rate per (trigger, method) that reaches ASR >= threshold
This is the number to actually report: e.g. "Random needs X% poison rate to reach 90% ASR on the word trigger; CBS needs Y%" -- a clean, quotable way to say CBS is less sample-efficient at installing the backdoor, without relying on a single shared rate that saturates one method and not the other.

In [9]:
def first_rate_reaching(df, trigger, method, threshold=SATURATION_ASR_THRESHOLD):
    sub = df[(df["trigger"] == trigger) & (df["method"] == method)].sort_values("poison_rate")
    hit = sub[sub["ASR"] >= threshold]
    return hit["poison_rate"].iloc[0] if len(hit) else None  # None = never reached threshold in this sweep

saturation_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        saturation_rows.append({
            "trigger": trigger, "method": method,
            f"first_rate_ASR>={SATURATION_ASR_THRESHOLD}": first_rate_reaching(sweep_df, trigger, method),
        })
saturation_summary_df = pd.DataFrame(saturation_rows)
saturation_summary_df

,trigger,method,first_rate_ASR>=0.9
0,word,random,0.0006
1,word,cbs,0.0050
2,sent,random,0.0004
3,sent,cbs,0.0020


## Save everything to xlsx

In [10]:
os.makedirs("./results", exist_ok=True)
with pd.ExcelWriter("./results/validation_results.xlsx", engine="openpyxl") as writer:
    multi_seed_df.to_excel(writer, sheet_name="multi_seed_raw", index=False)
    multi_seed_summary.to_excel(writer, sheet_name="multi_seed_summary")
    sweep_df.to_excel(writer, sheet_name="sweep", index=False)
    saturation_summary_df.to_excel(writer, sheet_name="saturation_summary", index=False)
print("saved ./results/validation_results.xlsx")

saved ./results/validation_results.xlsx
